# Module B2: Sentiment Analysis & Chatbot Intent Training
Build the NLP engine by processing the local `../data/reviews.csv` file (manually verified E-Commerce dataset) and `../data/intents.json`. Train a Logistic Regression model for sentiment analysis and a Naive Bayes model for the chatbot intents fallback. Save both to `app/models/`.

In [ ]:
import os
import json
import re
import joblib
import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Ensure lemmatization corpora are secured locally before running pipeline loops
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

# Dynamic path resolution: checks if execution is inside 'notebooks' or the project root
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == "notebooks":
    MODELS_DIR = os.path.join("..", "app", "models")
    DATA_DIR = os.path.join("..", "data")
else:
    MODELS_DIR = os.path.join("app", "models")
    DATA_DIR = os.path.join("data")

os.makedirs(MODELS_DIR, exist_ok=True)
reviews_path = os.path.join(DATA_DIR, "reviews.csv")
intents_path = os.path.join(DATA_DIR, "intents.json")

print(f"[INFO] Target Reviews Path resolved to: {os.path.abspath(reviews_path)}")
print(f"[INFO] Target Models Directory resolved to: {os.path.abspath(MODELS_DIR)}")

# Initialize the lemmatizer to achieve identical processing tokens to service layer
lemmatizer = WordNetLemmatizer()

def clean_and_lemmatize(text):
    """Fulfills Module B1: Standardizes textual tokens to match live inference arrays."""
    if not isinstance(text, str):
        return ""
    # Lowercase and punctuation removal
    cleaned = re.sub(r'[^\w\s]', '', text.lower()).strip()
    # Simple whitespace split tokenization + basic stopword filtering subset
    stopwords = {'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'has', 'he',
                 'in', 'is', 'it', 'its', 'of', 'on', 'that', 'the', 'to', 'was', 'were', 'will', 'with'}
    tokens = [t for t in cleaned.split() if t not in stopwords]
    # Lemmatization core loop transformation
    lemmatized = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(lemmatized)

# ==========================================
# 1. Sentiment Engine from local reviews.csv
# ==========================================
if os.path.exists(reviews_path):
    df_raw = pd.read_csv(reviews_path).dropna(subset=['Review Text'])
    
    # Process inputs using the synchronized text preprocessing pipeline
    df_processed = pd.DataFrame({
        'text': df_raw['Review Text'].apply(clean_and_lemmatize),
        'sentiment': df_raw['Rating'].apply(lambda r: 'Positive' if r >= 4 else ('Neutral' if r == 3 else 'Negative'))
    })
    
    sentiment_vectorizer = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))
    X_sent = sentiment_vectorizer.fit_transform(df_processed['text'])
    
    sentiment_clf = LogisticRegression(max_iter=500)
    sentiment_clf.fit(X_sent, df_processed['sentiment'])
    
    # FIXED: Saved as a unified dictionary matching your backend app's loading syntax
    joblib.dump({
        'vectorizer': sentiment_vectorizer,
        'model': sentiment_clf
    }, os.path.join(MODELS_DIR, "sentiment_model.pkl"))
    
    print("[SUCCESS] Review Sentiment compound model saved successfully.")
else:
    print(f"[ERROR] Could not find reviews.csv at {os.path.abspath(reviews_path)}")

# ==========================================
# 2. Chatbot Engine from local intents.json
# ==========================================
if os.path.exists(intents_path):
    with open(intents_path, 'r', encoding='utf-8') as f:
        intents_data = json.load(f)
        
    if isinstance(intents_data, dict):
        intents_list = intents_data.get("intents", [])
    else:
        intents_list = intents_data
            
    patterns, labels = [], []
    for intent in intents_list:
        tag = intent["tag"]
        for pattern in intent["patterns"]:
            # Standardize intent template definitions identically
            patterns.append(clean_and_lemmatize(pattern))
            labels.append(tag)
            
    chatbot_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    X_chat = chatbot_vectorizer.fit_transform(patterns)
    
    chatbot_clf = MultinomialNB(alpha=0.5)
    chatbot_clf.fit(X_chat, labels)
    
    # FIXED: Remapped key values to 'vectorizer' and 'model' to clear the backend KeyError
    joblib.dump({
        'vectorizer': chatbot_vectorizer,
        'model': chatbot_clf,
        'intents_data': intents_data
    }, os.path.join(MODELS_DIR, "chatbot_model.pkl"))
    
    print("[SUCCESS] Chatbot Intent compound model saved successfully.")
else:
    print(f"[ERROR] Could not find intents.json at {os.path.abspath(intents_path)}")

[INFO] Target Reviews Path resolved to: c:\smart-retail-ai\data\reviews.csv
[INFO] Target Models Directory resolved to: c:\smart-retail-ai\app\models
[SUCCESS] Review Sentiment compound model saved successfully.
[SUCCESS] Chatbot Intent compound model saved successfully.


: 